# Milestone 1 · Exploratory Data Analysis
**AIE231 Neural Networks · Course project** · Dataset: *Beijing Multi-Site Air Quality* (UCI #501)

**Goal of the project.** Forecast the PM2.5 concentration at a Beijing monitoring station **6 hours ahead**, from the last hours of pollution and weather at that station.

**Goal of this notebook.** Understand the data before modelling: its structure, missing values, the target, time patterns, the effect of weather, the differences between stations and suspicious values. Every section ends with what we saw, and the notebook ends with a list of **findings → decisions** for the cleaning step.

**Source.** Chen, S. (2017). *Beijing Multi-Site Air Quality* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5RK5G (licence CC BY 4.0). Paper: Zhang et al. (2017), *Cautionary tales on air-quality improvement in Beijing*, Proc. R. Soc. A.

## 0. Setup
### 0.1 Libraries
pandas/NumPy for the table, Matplotlib/Seaborn for plots. Nothing here trains a model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

### 0.2 Paths and constants
The raw files live in `data/raw/` (ignored by Git, see `.gitignore`). We use a **relative path** so the notebook runs on any teammate's machine (the spec forbids hard-coded paths).

`EPA_BANDS` are the US EPA 24-hour PM2.5 categories (µg/m³). We use them only to describe how dangerous an hour is.

In [ ]:
RAW_DIR = Path("data/raw")

POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
WEATHER = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
NUMERIC = POLLUTANTS + WEATHER
TARGET = "PM2.5"

EPA_BANDS = [0, 12, 35.4, 55.4, 150.4, 250.4, np.inf]
EPA_LABELS = ["Good", "Moderate", "Sensitive groups", "Unhealthy", "Very unhealthy", "Hazardous"]

files = sorted(RAW_DIR.glob("PRSA_Data_*.csv"))
print(len(files), "station files found")

## 1. Loading the data
### 1.1 Read and stack the 12 station files
Each CSV is one station. We read all 12 and stack them into one table, then build a real `datetime` column from `year, month, day, hour`, and sort by station and time.

In [ ]:
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)

print(df.shape)
df.head()

### 1.2 Column types and summary statistics
`info()` shows each column's type and how many non-missing values it has; `describe()` gives the range of every numeric column.

In [ ]:
df.info()

In [ ]:
df[NUMERIC].describe().T.round(1)

**What we see.** 420,768 rows × 18 original columns. Everything is numeric except `wd` (wind direction, text) and `station` (text). The counts are below 420,768 for every pollutant and weather column, so there are missing values. Some maxima look suspicious: PM2.5 = 999, CO = 10,000, O3 = 1,071 (checked in Section 8).

## 2. Structure: is every hour there exactly once?
### 2.1 Rows per station, duplicates and the hourly timeline
A forecasting model reads the data as a sequence, so we need **one row per station per hour**, with no duplicates and no missing hours.

In [ ]:
rows_per_station = df.groupby("station").size()
print(rows_per_station.to_string())

expected_hours = pd.date_range(df["datetime"].min(), df["datetime"].max(), freq="h")
print("\nHours from first to last timestamp:", len(expected_hours))
print("Duplicate (station, datetime) rows:", df.duplicated(["station", "datetime"]).sum())
print("Time span:", df["datetime"].min(), "->", df["datetime"].max())

**What we see.** Every station has exactly 35,064 rows, which is every hour from 1 Mar 2013 to 28 Feb 2017, and there are no duplicates. The **timeline itself is complete**; the problem is missing *values* inside it, not missing rows.

## 3. Missing values
### 3.1 Percentage missing per column

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

ax = missing.plot.barh(figsize=(7, 3.5), color="tab:red")
ax.invert_yaxis()
ax.set_xlabel("% of rows missing")
ax.set_title("Missing values per column")
for i, v in enumerate(missing):
    ax.text(v + 0.05, i, f"{v:.2f}%", va="center", fontsize=9)
plt.show()

print("Rows with at least one missing value: %.1f%%" % (df.isna().any(axis=1).mean() * 100))

### 3.2 Missing values per station
Are the gaps spread evenly, or does one station have most of them?

In [ ]:
miss_station = df.groupby("station")[POLLUTANTS + WEATHER + ["wd"]].apply(lambda g: g.isna().mean() * 100)

plt.figure(figsize=(10, 4.5))
sns.heatmap(miss_station, annot=True, fmt=".1f", cmap="Reds", cbar_kws={"label": "% missing"})
plt.title("% missing per station and column")
plt.show()

### 3.3 Outages: all pollutants missing at once
If all 6 pollutants are missing in the same hour, the station was most likely offline (an outage), not just one faulty sensor.

In [ ]:
all_poll_missing = df[POLLUTANTS].isna().all(axis=1)
print("Hours with all 6 pollutants missing:", all_poll_missing.sum())
print(df.loc[all_poll_missing].groupby("station").size().sort_values(ascending=False).to_string())

### 3.4 How long are the gaps in the target?
A 1-hour gap can be filled from its neighbours; a 2-week gap cannot. `gap_lengths` walks through each station's PM2.5 series and measures every run of consecutive missing hours.

In [ ]:
def gap_lengths(series):
    """Return the length (in hours) of every run of consecutive missing values."""
    is_na = series.isna().to_numpy()
    lengths, run = [], 0
    for missing_now in is_na:
        if missing_now:
            run += 1
        elif run:
            lengths.append(run)
            run = 0
    if run:
        lengths.append(run)
    return lengths

gaps = pd.Series([g for _, s in df.groupby("station")[TARGET] for g in gap_lengths(s)])
gap_groups = pd.cut(gaps, bins=[0, 3, 24, np.inf], labels=["1-3 h", "4-24 h", "> 24 h"])

summary = pd.DataFrame({"gaps": gap_groups.value_counts().sort_index(),
                        "missing hours": gaps.groupby(gap_groups, observed=False).sum()})
print(summary)
print("Longest gap:", gaps.max(), "hours (= %.1f days)" % (gaps.max() / 24))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(gaps, bins=np.arange(1, 50), color="tab:red", edgecolor="white")
ax.set_yscale("log")
ax.set_xlabel("gap length (hours, cut at 48)")
ax.set_ylabel("number of gaps (log)")
ax.set_title("PM2.5 gap lengths, all stations")
plt.show()

### 3.5 When do the gaps happen?
Each dot is a missing PM2.5 hour, one line per station. Long horizontal bars are outages.

In [ ]:
stations = sorted(df["station"].unique())
fig, ax = plt.subplots(figsize=(11, 4))
for i, st in enumerate(stations):
    t = df.loc[(df["station"] == st) & df[TARGET].isna(), "datetime"]
    ax.plot(t, np.full(len(t), i), "|", color="tab:red", markersize=8)
ax.set_yticks(range(len(stations)))
ax.set_yticklabels(stations)
ax.set_title("Missing PM2.5 hours over time")
plt.show()

**What we see.**
* Missing values are small overall (PM2.5 ≈ 2.1 %, CO ≈ 4.9 % is the worst), but about **9 % of rows** miss *something*.
* There are about 5,000 **outage hours** where all pollutants are missing together.
* Most PM2.5 gaps are **short (1-3 h)**, which interpolation handles well. A few dozen are **longer than a day**, the longest about 2 weeks; filling those would mean inventing data.
* Gaps appear at the same time at several stations, which points to network-wide outages.

## 4. The target: PM2.5
### 4.1 Distribution

In [ ]:
pm = df[TARGET].dropna()
print("mean %.1f | median %.1f | skew %.2f | max %.0f" % (pm.mean(), pm.median(), pm.skew(), pm.max()))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].hist(pm, bins=100, color="tab:blue")
ax[0].set_title("PM2.5 (µg/m³)")
ax[1].hist(np.log1p(pm), bins=100, color="tab:green")
ax[1].set_title("log(1 + PM2.5)")
plt.show()

### 4.2 How dangerous are the hours? (EPA categories)
This is our version of "class imbalance" for a regression task: the dangerous hours are the ones that matter most, and they are the rarest.

In [ ]:
bands = pd.cut(pm, bins=EPA_BANDS, labels=EPA_LABELS)
band_share = bands.value_counts(normalize=True).reindex(EPA_LABELS) * 100

colors = ["#3fae6a", "#c9b43a", "#e08a3c", "#d9534f", "#9b59b6", "#7b2d3b"]
ax = band_share.plot.bar(figsize=(8, 3.4), color=colors, rot=20)
ax.set_ylabel("% of hours")
ax.set_title("PM2.5 hours by EPA category")
for i, v in enumerate(band_share):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)
plt.show()

**What we see.** PM2.5 is strongly **right-skewed** (median 55, mean 80, skew ≈ 2): many moderate hours and a long tail of extreme ones up to 999. The log transform makes the shape much more symmetric, which usually helps a network train. About **15 % of hours are above 150** (Unhealthy or worse) and about **4.5 % above 250**: rare but exactly the hours a forecast must not miss.

## 5. Time patterns
### 5.1 The whole period (daily average, all stations)

In [ ]:
daily = df.set_index("datetime").groupby("station")[TARGET].resample("D").mean().unstack(0)

fig, ax = plt.subplots(figsize=(12, 3.6))
ax.plot(daily.index, daily.mean(axis=1), lw=0.8)
ax.plot(daily.index, daily.mean(axis=1).rolling(30, center=True).mean(), lw=2, color="black", label="30-day mean")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_title("Daily mean PM2.5, average of the 12 stations")
ax.legend()
plt.show()

print(df.groupby("year")[TARGET].mean().round(1).to_string())

### 5.2 Season (month) and daily cycle (hour)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
sns.boxplot(data=df, x="month", y=TARGET, ax=ax[0], showfliers=False, color="tab:blue")
ax[0].set_title("PM2.5 by month (outliers hidden)")
df.groupby("hour")[TARGET].mean().plot(ax=ax[1], marker="o")
ax[1].set_title("Average PM2.5 by hour of day")
ax[1].set_ylabel("PM2.5 (µg/m³)")
plt.show()

In [ ]:
pivot = df.pivot_table(index="hour", columns="month", values=TARGET, aggfunc="mean")
plt.figure(figsize=(10, 5))
sns.heatmap(pivot, cmap="rocket_r", cbar_kws={"label": "mean PM2.5"})
plt.title("Mean PM2.5 by hour and month")
plt.show()

**What we see.** The worst months are **December-March** (heating season, about 95-105 µg/m³) and the cleanest is **August** (about 53). Within a day, PM2.5 is highest **around midnight** and lowest in the **late morning**. The hour × month map shows both effects together: winter nights are the worst. Year to year there is no clear downward trend (2017 is only Jan-Feb, which are winter months). These cycles mean the model needs to know **the time of day and the season**.

## 6. Stations
### 6.1 Average level per station

In [ ]:
order = df.groupby("station")[TARGET].mean().sort_values().index
plt.figure(figsize=(10, 3.6))
sns.barplot(data=df, x="station", y=TARGET, order=order, errorbar=None, color="tab:blue")
plt.xticks(rotation=40, ha="right")
plt.title("Mean PM2.5 per station")
plt.show()

### 6.2 Do the stations move together?
Correlation between the stations' hourly PM2.5. High values mean the whole city rises and falls together.

In [ ]:
hourly_wide = df.pivot(index="datetime", columns="station", values=TARGET)
plt.figure(figsize=(9, 7))
sns.heatmap(hourly_wide.corr(), annot=True, fmt=".2f", cmap="viridis", vmin=0.7, vmax=1)
plt.title("Correlation of hourly PM2.5 between stations")
plt.show()

**What we see.** Suburban stations (Dingling, Huairou, Changping) are cleaner than the central ones (Dongsi, Wanshouxigong, Nongzhanguan). But the stations are **very strongly correlated** with each other: pollution is a city-wide event driven by weather. That makes "train on some stations, test on stations the model never saw" a realistic generalization test.

## 7. Weather and pollution
### 7.1 Wind direction

In [ ]:
wd_order = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
wd_mean = df.groupby("wd")[TARGET].mean().reindex(wd_order)

angles = np.deg2rad(np.arange(0, 360, 22.5))
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="polar")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.bar(angles, wd_mean.values, width=np.deg2rad(20), color="tab:orange", alpha=0.8)
ax.set_xticks(angles)
ax.set_xticklabels(wd_order)
ax.set_title("Mean PM2.5 by wind direction (wind coming FROM)")
plt.show()

### 7.2 Wind speed, humidity and rain
Numbers are grouped into bins so the trend is readable.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))

ws_bins = pd.cut(df["WSPM"], [0, 1, 2, 3, 4, 6, 14], include_lowest=True)
df.groupby(ws_bins, observed=True)[TARGET].mean().plot.bar(ax=ax[0], rot=30, color="tab:blue")
ax[0].set_title("PM2.5 by wind speed (m/s)")

spread = df["TEMP"] - df["DEWP"]           # small spread = humid air
sp_bins = pd.cut(spread, [-1, 2, 5, 10, 15, 20, 30, 60])
df.groupby(sp_bins, observed=True)[TARGET].mean().plot.bar(ax=ax[1], rot=30, color="tab:green")
ax[1].set_title("PM2.5 by TEMP - DEWP (°C)\nsmall = humid")

rain = np.where(df["RAIN"] > 0, "raining", "dry")
df.groupby(rain)[TARGET].mean().plot.bar(ax=ax[2], rot=0, color="tab:purple")
ax[2].set_title("PM2.5: rain vs no rain")
plt.tight_layout()
plt.show()

**What we see.** Wind is the strongest weather signal. **North-west winds** bring clean air from the mountains (about 50 µg/m³), while **east and south-east winds** bring polluted air from the plain (about 100). **Calm air** (< 1 m/s) traps pollution; strong wind clears it. **Humid air** (small TEMP - DEWP) goes with high PM2.5, typical of winter smog, and rain washes the air. So the weather columns are real inputs for the model, not decoration.

## 8. Correlations and memory
### 8.1 Correlation between all numeric columns

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[NUMERIC].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

### 8.2 Autocorrelation: how much does the past tell us about the future?
Correlation of PM2.5 with itself *k* hours earlier, at one station. This decides how much history the model should read (window length) and how far ahead a forecast is still meaningful (horizon).

In [ ]:
s = df.loc[df["station"] == "Dongsi"].set_index("datetime")[TARGET]
lags = range(1, 73)
acf = [s.autocorr(lag) for lag in lags]

plt.figure(figsize=(9, 3.4))
plt.plot(lags, acf, marker=".")
for lag in (1, 6, 24):
    plt.axvline(lag, color="grey", ls=":")
    plt.text(lag + 0.5, acf[lag - 1] + 0.02, f"lag {lag}: {acf[lag - 1]:.2f}")
plt.xlabel("lag (hours)")
plt.ylabel("autocorrelation")
plt.title("PM2.5 autocorrelation (Dongsi)")
plt.show()

**What we see.** PM2.5 moves with **PM10 (0.88)**, **CO (0.79)** and **NO2 (0.67)**: the same sources. Wind speed is negative (−0.27). Temperature, dew point and pressure are strongly related to each other (all three follow the seasons), which is redundant information.

The autocorrelation is **0.97 after 1 hour**, about **0.76 after 6 hours** and about **0.4 after 24 hours**. One hour ahead, "same as now" is almost perfect, so a model can barely add anything. **Six hours ahead** the past still helps a lot but no longer gives the answer, which is the horizon where a recurrent network can show its value. Memory fades over about a day, so a **24-hour input window** is a sensible starting point.

## 9. Outliers and impossible values
### 9.1 Boxplots of every pollutant

In [ ]:
fig, axes = plt.subplots(1, len(POLLUTANTS), figsize=(14, 3.6))
for ax, col in zip(axes, POLLUTANTS):
    ax.boxplot(df[col].dropna(), flierprops={"markersize": 1, "alpha": 0.3})
    ax.set_title(col)
    ax.set_xticks([])
plt.tight_layout()
plt.show()

### 9.2 Values that cannot be true
* **PM10 < PM2.5:** PM2.5 is *part of* PM10, so PM10 can never be smaller. Small differences can be measurement noise; large ones are sensor errors.
* **Instrument caps:** values sitting exactly at a round maximum (999, 10,000) look like the instrument's upper limit, not a real reading.

In [ ]:
diff = df["PM10"] - df["PM2.5"]
print("PM10 < PM2.5           :", (diff < 0).sum(), "rows")
print("PM10 < PM2.5 by > 10   :", (diff < -10).sum(), "rows")
print("PM10 < PM2.5 by > 50   :", (diff < -50).sum(), "rows")
print()
print("PM2.5 == 999 :", (df["PM2.5"] == 999).sum())
print("PM10  == 999 :", (df["PM10"] == 999).sum())
print("CO == 10000  :", (df["CO"] == 10000).sum())
print("O3 > 500     :", (df["O3"] > 500).sum())

### 9.3 Real extremes: Chinese New Year fireworks
Before deciding that a spike is an error, check whether it has a real cause. Chinese New Year's Eve fireworks are a well-known pollution event.

In [ ]:
cny_eve = ["2014-01-30", "2015-02-18", "2016-02-07"]
for day in cny_eve:
    start = pd.Timestamp(day) + pd.Timedelta(hours=18)
    window = df[(df["datetime"] >= start) & (df["datetime"] < start + pd.Timedelta(hours=12))]
    peak = window.loc[window[TARGET].idxmax()]
    print(f"{day} evening: max PM2.5 = {peak[TARGET]:.0f} at {peak['station']} {peak['datetime']:%H:%M}")

**What we see.** Every pollutant has a long tail of high values. Most of it is **real** (smog episodes, fireworks), and those spikes are exactly what the forecast should warn about, so they must **not** be removed just for being large. What looks wrong is narrower: about **21,000 rows with PM10 < PM2.5** (about 2,300 by more than 50, clearly errors) and a few readings stuck at **instrument caps** (CO = 10,000, values of 999).

## 10. Findings → decisions
Each finding leads to a decision for the cleaning step (Milestone 1, part 2) or for modelling later.

| # | Finding | Decision |
|---|---|---|
| 1 | Complete hourly timeline, no duplicates | Keep the time index as is; no rows to add or drop for structure |
| 2 | Short PM2.5 gaps (1-3 h) are the large majority | Fill them by **time interpolation inside each station** (never across stations) |
| 3 | A few long gaps (> 24 h, up to ~2 weeks) and ~5,000 outage hours | **Do not invent** these stretches; mark them so input windows never cross them |
| 4 | Weather columns are almost complete; `wd` is text with 16 values | Fill the few weather gaps by interpolation / nearest value; encode `wd` as an **angle (sin, cos)** instead of 16 one-hot columns |
| 5 | PM2.5 is right-skewed (skew ≈ 2) | Try a **log(1 + PM2.5)** target in modelling and compare with the raw target |
| 6 | ~15 % of hours are Unhealthy or worse | Report error on high-pollution hours separately; consider a **weighted loss** for spikes |
| 7 | Strong daily and seasonal cycles | Add **cyclical time features** (hour and month as sin/cos) |
| 8 | Wind direction, wind speed, humidity and rain clearly affect PM2.5 | Keep all weather columns as model inputs |
| 9 | PM2.5 strongly correlated with PM10, CO, NO2 | Keep them as inputs (past values only, so no leakage) |
| 10 | TEMP, DEWP, PRES strongly correlated with each other | Keep for now; test removing them in an ablation |
| 11 | Autocorrelation 0.97 at 1 h, ~0.76 at 6 h, ~0.4 at 24 h | Forecast **6 h ahead**; start with a **24 h input window**; "next = now" is the trivial baseline |
| 12 | Stations differ in level but move together | Add the station as an input; use **unseen stations** as a generalization test |
| 13 | PM10 < PM2.5 in ~21,000 rows | Treat the large violations as **sensor errors in PM10** and set them to missing (PM2.5, the target, is kept) |
| 14 | Values at instrument caps (CO = 10,000, 999) | Set clear cap values to missing and fill like other gaps |
| 15 | Real extreme events (fireworks, smog) | **Keep** them; they are what the model must predict |
| 16 | Data is a time series | Later: split **in time order** (no shuffling) and fit scalers on the training years only |